## Extraction of data

In [1]:
import mysql.connector
import pandas as pd

# Connect to the MySQL database
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='root',
    database='ecofriendlypr'
)

# Define SQL queries for each table
queries = {
    'users': 'SELECT * FROM users;',
    'products': 'SELECT * FROM products;',
    'reviews': 'SELECT * FROM reviews;'
}

# Execute each query and store the results in a dictionary of DataFrames
dataframes = {}
for table, query in queries.items():
    dataframes[table] = pd.read_sql(query, conn)
    print(f"\n--- {table.upper()} TABLE ---")
    print(dataframes[table])

# Close the connection
conn.close()



--- USERS TABLE ---
    user_id  product_id
0       100        1051
1       101        1092
2       102        1014
3       103        1071
4       104        1060
..      ...         ...
95      195        1084
96      196        1079
97      197        1081
98      198        1052
99      199        1023

[100 rows x 2 columns]

--- PRODUCTS TABLE ---
    product_id                 product_name     brand     category  price  \
0         1000            Bamboo Toothbrush  EcoBrand   Toothbrush  45.67   
1         1001  Recycled Plastic Toothbrush  EcoBrand   Toothbrush  24.41   
2         1002        Cornstarch Toothbrush  EcoBrand   Toothbrush  16.19   
3         1003          Silicone Toothbrush  EcoBrand   Toothbrush  48.35   
4         1004  Charcoal-infused Toothbrush  EcoBrand   Toothbrush  15.17   
..         ...                          ...       ...          ...    ...   
95        1095       Flax Fiber Phone Cases  EcoBrand  Phone Cases  25.86   
96        1096  Recycled Ru

C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\3279310228.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dataframes[table] = pd.read_sql(query, conn)
C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\3279310228.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dataframes[table] = pd.read_sql(query, conn)
C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\3279310228.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dataframes[table] = pd.read_sql(query, conn)


## Filtering

## Content based filtering

In [15]:
# 📦 Imports
import pandas as pd
import mysql.connector
import panel as pn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pn.extension()

# 🔌 MySQL Connection
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="ecofriendlypr"
)

# 📄 Load Product Data
df = pd.read_sql("SELECT * FROM products;", conn)

# 🎛️ Panel Widgets (start with empty selection)
category_select = pn.widgets.Select(name='Select Category', options=[""] + sorted(df['category'].dropna().unique().tolist()))
product_select = pn.widgets.Select(name='Select Product', options=[""])
recommendation_output = pn.pane.DataFrame(height=300, sizing_mode='stretch_width')

# 🔄 Update product dropdown when category is selected
def update_product_dropdown(event=None):
    selected_category = category_select.value
    if selected_category:
        filtered = df[df['category'] == selected_category]
        product_list = filtered['product_name'].dropna().tolist()
        product_select.options = [""] + product_list
        product_select.value = ""
    else:
        product_select.options = [""]
        product_select.value = ""

# 🤖 Recommend similar products
def recommend_products(event=None):
    selected_category = category_select.value
    product_name = product_select.value

    if not selected_category or not product_name:
        recommendation_output.object = pd.DataFrame()
        return

    df_filtered = df[df['category'] == selected_category].reset_index(drop=True)

    df_filtered['combined_features'] = (
        df_filtered['product_name'].fillna('') + " " +
        df_filtered['category'].fillna('') + " " +
        df_filtered['brand'].fillna('')
    )

    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(df_filtered['combined_features'])
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    if product_name not in df_filtered['product_name'].values:
        recommendation_output.object = pd.DataFrame({'Message': ["❌ Product not found in the selected category."]})
        return

    idx = df_filtered[df_filtered['product_name'] == product_name].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:6]
    product_indices = [i[0] for i in sim_scores]
    recommendations = df_filtered.iloc[product_indices][['product_id', 'product_name', 'category', 'brand']]

    recommendation_output.object = recommendations

    # 💾 Log to MySQL
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS contentbasedrecomm;")
    cursor.execute('''
        CREATE TABLE contentbasedrecomm (
            id INT AUTO_INCREMENT PRIMARY KEY,
            input_product VARCHAR(255),
            recommended_product VARCHAR(255)
        );
    ''')
    for rec in recommendations['product_name']:
        cursor.execute(
            "INSERT INTO contentbasedrecomm (input_product, recommended_product) VALUES (%s, %s);",
            (product_name, rec)
        )
    conn.commit()

# 🔗 Link widgets
category_select.param.watch(update_product_dropdown, 'value')
product_select.param.watch(recommend_products, 'value')

# 🧱 Layout
dashboard = pn.Column(
    "## 🌿 Content-Based Product Recommender",
    category_select,
    product_select,
    pn.pane.Markdown("### 🧠 Recommended Products:"),
    recommendation_output
)

dashboard


C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\3670955652.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM products;", conn)


Column
    [0] Markdown(str)
    [1] Select(options=['', 'Bag', 'Clothing', ...])
    [2] Select(options=[''])
    [3] Markdown(str)
    [4] DataFrame(None, height=300, sizing_mode='stretch_width')

## Colloborative filtering

In [ ]:
##Item-based collaborative filtering using co-occurrence counts (not product content).

 In Recommender Systems:
Co-occurrence counts how often two products are interacted with by the same user.

In [16]:
import panel as pn
import pandas as pd
import mysql.connector

pn.extension()

# 🔌 MySQL Connection
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="ecofriendlypr"
)

# 📥 Load Data
products = pd.read_sql("SELECT * FROM products;", conn)
reviews = pd.read_sql("SELECT user_id, product_name FROM reviews;", conn)

# 📊 Co-occurrence Matrix Construction
cooccurrence_matrix = {}

for user_id, group in reviews.groupby('user_id'):
    user_products = group['product_name'].unique()
    for i in range(len(user_products)):
        for j in range(len(user_products)):
            if i != j:
                item_i = user_products[i]
                item_j = user_products[j]
                cooccurrence_matrix.setdefault(item_i, {})
                cooccurrence_matrix[item_i][item_j] = cooccurrence_matrix[item_i].get(item_j, 0) + 1

# 🧠 Recommendation Function
def get_similar_products(product_name, top_n=5):
    related = cooccurrence_matrix.get(product_name, {})
    sorted_related = sorted(related.items(), key=lambda x: x[1], reverse=True)
    return [x[0] for x in sorted_related[:top_n]]

# 📦 Save recommendations to database
def save_to_db(product_name, recommended_products):
    cursor = conn.cursor()
    
    # Create table if not exists
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS collaborative_recommen (
            id INT AUTO_INCREMENT PRIMARY KEY,
            product_name VARCHAR(255),
            recommended_product VARCHAR(255)
        );
    """)
    
    # Insert recommendations
    for rec in recommended_products:
        cursor.execute("""
            INSERT INTO collaborative_recommen (product_name, recommended_product)
            VALUES (%s, %s);
        """, (product_name, rec))
    
    conn.commit()

# 🎛️ UI Widgets
category_select = pn.widgets.Select(name='Select Category', options=[""] + sorted(products['category'].dropna().unique().tolist()))
product_select = pn.widgets.Select(name='Select Product', options=[""])

recommendation_output = pn.pane.DataFrame(height=300, sizing_mode="stretch_width")

# 🔄 Update product dropdown based on selected category
def update_products(event):
    selected_cat = category_select.value
    if selected_cat:
        filtered = products[products['category'] == selected_cat]
        product_options = filtered['product_name'].dropna().unique().tolist()
        product_select.options = [""] + product_options
        product_select.value = ""  # Clear selection
    else:
        product_select.options = [""]
        product_select.value = ""

category_select.param.watch(update_products, 'value')

# 🚀 Generate recommendations
def generate_recommendations(event):
    recommendation_output.object = pd.DataFrame()  # clear

    selected_product = product_select.value
    if selected_product:
        recommendations = get_similar_products(selected_product)
        if recommendations:
            df_result = products[products['product_name'].isin(recommendations)][
                ['product_id', 'product_name', 'category']
            ].reset_index(drop=True)
            recommendation_output.object = df_result
            save_to_db(selected_product, recommendations)
        else:
            recommendation_output.object = pd.DataFrame({'message': ['No recommendations found.']})

product_select.param.watch(generate_recommendations, 'value')

# 🧱 Layout
dashboard = pn.Column(
    "## 👥 Collaborative Filtering Recommender",
    category_select,
    product_select,
    pn.Spacer(height=20),
    recommendation_output
)

dashboard.servable()


C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\4047152191.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql("SELECT * FROM products;", conn)
C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\4047152191.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reviews = pd.read_sql("SELECT user_id, product_name FROM reviews;", conn)


Column
    [0] Markdown(str)
    [1] Select(options=['', 'Bag', 'Clothing', ...])
    [2] Select(options=[''])
    [3] Spacer(height=20)
    [4] DataFrame(None, height=300, sizing_mode='stretch_width')

In [17]:
import panel as pn
import pandas as pd
import mysql.connector
import re

pn.extension()

# MySQL connection
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="ecofriendlypr"
)

# Load tables
products = pd.read_sql("SELECT * FROM products", conn)
reviews = pd.read_sql("SELECT user_id, product_name FROM reviews", conn)  # Assuming product_name exists

# Define category co-purchase relationships
co_purchase_map = {
    "bag": ["notebook", "water bottle", "clothing"],
    "clothing": ["shoes", "toys", "bag"],
    "cutlery": ["straws"],
    "notebook": ["bag", "water bottle"],
    "phonecases": ["bag", "clothing", "toys"],
    "shoes": ["clothing", "bag", "toys"],
    "straws": ["cutlery", "water bottle"],
    "toothbrush": ["bag", "toys", "water bottle"],
    "toys": ["bag", "cutlery", "water bottle", "toothbrush"],
    "water bottle": ["bag", "notebook", "straws"]
}

# Material keywords
materials = ['wood', 'bamboo', 'stainless steel', 'glass', 'silicone', 'plastic',
             'copper', 'hemp', 'stone', 'cork', 'recycled', 'metal', 'cloth', 'leather', 'paper', 'wool']

# Extract matching material from a product name
def extract_material(product_name):
    name = product_name.lower()
    for material in materials:
        if material in name:
            return material
    return None

# Ensure the collaborative_recommendations table exists
def ensure_recommendations_table(cursor):
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS collaborative_recommendations (
            product_id INT,
            product_name TEXT,
            category TEXT
        )
    """)

# Get product recommendations
def get_recommendations(category, product_name):
    product_name = product_name.strip()
    
    # Match material from selected product
    material = extract_material(product_name)
    
    # Get related categories
    related_categories = co_purchase_map.get(category.lower(), [])
    
    # Products by related category
    category_products = products[
        products['category'].str.lower().isin([c.lower() for c in related_categories])
    ]
    
    # Products by shared material
    if material:
        material_products = products[
            products['product_name'].str.lower().str.contains(material)
        ]
    else:
        material_products = pd.DataFrame()

    # Combine and remove duplicates
    combined = pd.concat([category_products, material_products]).drop_duplicates()
    
    # Remove the selected product itself
    combined = combined[combined['product_name'] != product_name]

    # Shuffle and take top 7
    combined = combined.sample(frac=1).head(7)

    # Save to DB
    cursor = conn.cursor()
    ensure_recommendations_table(cursor)
    cursor.execute("DELETE FROM collaborative_recommendations")
    for _, row in combined.iterrows():
        cursor.execute(
            "INSERT INTO collaborative_recommendations (product_id, product_name, category) VALUES (%s, %s, %s)",
            (int(row['product_id']), row['product_name'], row['category'])
        )
    conn.commit()

    return combined.reset_index(drop=True)

# Panel Widgets
category_select = pn.widgets.Select(name='Select Category', options=[''] + sorted(products['category'].unique().tolist()))
product_select = pn.widgets.Select(name='Select Product', options=[''])

output_pane = pn.pane.DataFrame(width=700)

# Callback to update product dropdown based on category
def update_products(event):
    selected_cat = category_select.value
    if selected_cat:
        filtered = products[products['category'] == selected_cat]
        product_select.options = [''] + sorted(filtered['product_name'].unique().tolist())
    else:
        product_select.options = ['']
        output_pane.object = pd.DataFrame()

category_select.param.watch(update_products, 'value')

# Callback to show recommendations
def show_recommendations(event):
    if category_select.value and product_select.value:
        recs = get_recommendations(category_select.value, product_select.value)
        output_pane.object = recs
    else:
        output_pane.object = pd.DataFrame()

product_select.param.watch(show_recommendations, 'value')

# Layout
pn.Column(
    "### 👥 Collaborative Filtering Recommender",
    category_select,
    product_select,
    output_pane
).servable()


C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\1092959725.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql("SELECT * FROM products", conn)
C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\1092959725.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reviews = pd.read_sql("SELECT user_id, product_name FROM reviews", conn)  # Assuming product_name exists


Column
    [0] Markdown(str)
    [1] Select(options=['', 'Bag', 'Clothing', ...])
    [2] Select(options=[''])
    [3] DataFrame(None, width=700)

## Hybrid based filtering

In [18]:
import panel as pn
import pandas as pd
import mysql.connector
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pn.extension()

# MySQL connection
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="ecofriendlypr"
)

# Load product data
products = pd.read_sql("SELECT * FROM products", conn)

# Co-purchase category map
co_purchase_map = {
    "bag": ["notebook", "water bottle", "clothing"],
    "clothing": ["shoes", "toys", "bag"],
    "cutlery": ["straws"],
    "notebook": ["bag", "water bottle"],
    "phonecases": ["bag", "clothing", "toys"],
    "shoes": ["clothing", "bag", "toys"],
    "straws": ["cutlery", "water bottle"],
    "toothbrush": ["bag", "toys", "water bottle"],
    "toys": ["bag", "cutlery", "water bottle", "toothbrush"],
    "water bottle": ["bag", "notebook", "straws"]
}

# Material extraction keywords
materials = ['wood', 'bamboo', 'stainless steel', 'glass', 'silicone', 'plastic',
             'copper', 'hemp', 'stone', 'cork', 'recycled', 'metal', 'cloth', 'leather', 'paper', 'wool', 'cotton', 'linen', 'canvas']

def extract_material(product_name):
    name = product_name.lower()
    for material in materials:
        if material in name:
            return material
    return None

# Panel widgets
category_select = pn.widgets.Select(name='Select Category', options=[''] + sorted(products['category'].dropna().unique().tolist()))
product_select = pn.widgets.Select(name='Select Product', options=[''])
output_pane = pn.pane.DataFrame(width=800)

# Update product list
def update_products(event):
    selected_cat = category_select.value
    if selected_cat:
        filtered = products[products['category'] == selected_cat]
        product_select.options = [''] + sorted(filtered['product_name'].unique().tolist())
    else:
        product_select.options = ['']
        output_pane.object = pd.DataFrame()

category_select.param.watch(update_products, 'value')

# Hybrid recommender logic
def hybrid_recommend(event):
    category = category_select.value
    product_name = product_select.value.strip()
    
    if not category or not product_name:
        output_pane.object = pd.DataFrame()
        return
    
    # --- COLLABORATIVE FILTERING ---
    material = extract_material(product_name)
    related_categories = co_purchase_map.get(category.lower(), [])
    
    collaborative_df = products[
        products['category'].str.lower().isin([cat.lower() for cat in related_categories])
    ]
    
    if material:
        material_df = products[products['product_name'].str.lower().str.contains(material)]
        collaborative_df = pd.concat([collaborative_df, material_df])
    
    collaborative_df = collaborative_df[collaborative_df['product_name'] != product_name]
    collaborative_df['source'] = 'collaborative'

    # --- CONTENT-BASED FILTERING ---
    df_filtered = products[products['category'] == category].reset_index(drop=True)
    df_filtered['combined'] = (
        df_filtered['product_name'].fillna('') + " " +
        df_filtered['category'].fillna('') + " " +
        df_filtered['brand'].fillna('')
    )

    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(df_filtered['combined'])
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

    if product_name not in df_filtered['product_name'].values:
        output_pane.object = pd.DataFrame({'Message': ["❌ Product not found in content-based search."]})
        return

    idx = df_filtered[df_filtered['product_name'] == product_name].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:10]
    content_df = df_filtered.iloc[[i[0] for i in sim_scores]]
    content_df = content_df[content_df['product_name'] != product_name]
    content_df['source'] = 'content'

    # --- HYBRID MERGE ---
    all_recs = pd.concat([collaborative_df, content_df])
    all_recs = all_recs.drop_duplicates(subset='product_name')

    # Prioritize shared results
    score = all_recs['source'].value_counts()
    all_recs['score'] = all_recs['product_name'].map(
        all_recs.groupby('product_name')['source'].apply(lambda x: len(set(x)))
    )
    all_recs = all_recs.sort_values(by='score', ascending=False)

    final_recs = all_recs[['product_id', 'product_name', 'category', 'brand', 'price', 'availability']].head(7)

    output_pane.object = final_recs

    # Save to MySQL
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS hybrid_recommendations;")
    cursor.execute("""
        CREATE TABLE hybrid_recommendations (
            id INT AUTO_INCREMENT PRIMARY KEY,
            input_product VARCHAR(255),
            recommended_product VARCHAR(255)
        );
    """)
    for rec in final_recs['product_name']:
        cursor.execute(
            "INSERT INTO hybrid_recommendations (input_product, recommended_product) VALUES (%s, %s);",
            (product_name, rec)
        )
    conn.commit()

product_select.param.watch(hybrid_recommend, 'value')

# Layout
pn.Column(
    "## 🔁 Hybrid Recommender (Content + Collaborative)",
    category_select,
    product_select,
    pn.pane.Markdown("### 🎯 Recommended Products:"),
    output_pane
).servable()


C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\946760329.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql("SELECT * FROM products", conn)


Column
    [0] Markdown(str)
    [1] Select(options=['', 'Bag', 'Clothing', ...])
    [2] Select(options=[''])
    [3] Markdown(str)
    [4] DataFrame(None, width=800)

## Price based filtering

In [19]:
import panel as pn
import pandas as pd
import mysql.connector
import numpy as np

pn.extension()

# MySQL connection
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="ecofriendlypr"
)

# Load product data
products = pd.read_sql("SELECT * FROM products", conn)

# --- Extract unique categories ---
categories = [''] + sorted(products['category'].dropna().unique().tolist())

# --- Generate dynamic price ranges ---
def get_price_ranges(df, bins=5):
    prices = df['price'].dropna()
    if prices.empty:
        return []
    labels = []
    bin_edges = np.linspace(prices.min(), prices.max(), bins + 1)
    for i in range(len(bin_edges)-1):
        start = round(bin_edges[i], 2)
        end = round(bin_edges[i+1], 2)
        labels.append(f"₹{start} - ₹{end}")
    return labels

price_ranges = get_price_ranges(products)

# --- Panel Widgets ---
category_select = pn.widgets.Select(name="Select Category", options=categories)
price_select = pn.widgets.Select(name="Select Price Range", options=[''] + price_ranges)
recommendation_output = pn.pane.DataFrame(height=300, sizing_mode='stretch_width')

# --- Filtering logic ---
def filter_products(event=None):
    selected_category = category_select.value
    selected_price_range = price_select.value

    if not selected_category or not selected_price_range:
        recommendation_output.object = pd.DataFrame()
        return

    # Parse price range
    match = [float(s.replace('₹','').strip()) for s in selected_price_range.split('-')]
    low, high = match[0], match[1]

    filtered = products[(products['category'] == selected_category) &
                        (products['price'] >= low) & (products['price'] <= high)]

    if filtered.empty:
        recommendation_output.object = pd.DataFrame({'Message': ["❌ No products found in this range."]})
        return

    recommendation_output.object = filtered[['product_id', 'product_name', 'category', 'brand', 'price', 'availability']]

    # --- Store in MySQL table ---
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS price_based_recommendations")
    cursor.execute('''
        CREATE TABLE price_based_recommendations (
            product_id INT,
            product_name TEXT,
            category TEXT,
            brand TEXT,
            price FLOAT,
            availability TEXT
        )
    ''')
    for _, row in filtered.iterrows():
        cursor.execute('''
            INSERT INTO price_based_recommendations
            (product_id, product_name, category, brand, price, availability)
            VALUES (%s, %s, %s, %s, %s, %s)
        ''', (int(row['product_id']), row['product_name'], row['category'], row['brand'], float(row['price']), row['availability']))
    conn.commit()

# --- Link dropdowns to filtering logic ---
category_select.param.watch(filter_products, 'value')
price_select.param.watch(filter_products, 'value')

# --- Layout ---
dashboard = pn.Column(
    "### 💰 Price-Based Product Filtering",
    category_select,
    price_select,
    pn.pane.Markdown("### 🛍️ Filtered Products:"),
    recommendation_output
)

# Show dashboard
dashboard.servable()

C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\3007519252.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products = pd.read_sql("SELECT * FROM products", conn)


Column
    [0] Markdown(str)
    [1] Select(options=['', 'Bag', 'Clothing', ...])
    [2] Select(options=['', '₹5.06 - ₹13.98', ...])
    [3] Markdown(str)
    [4] DataFrame(None, height=300, sizing_mode='stretch_width')

## Review based filtering

In [23]:
!pip install textblob


   ---------------------------------------- 0.0/624.3 kB ? eta -:--:--
   ---------------------------------------- 624.3/624.3 kB 3.9 MB/s eta 0:00:00


In [24]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [20]:
import pandas as pd
import panel as pn 
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import mysql.connector

pn.extension()

# 🔌 Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="ecofriendlypr"
)

# 📥 Load data directly from MySQL
query = "SELECT user_id, product_name AS `Product Name`, unique_reviews FROM reviews WHERE unique_reviews IS NOT NULL;"
df = pd.read_sql(query, conn)

# 🧠 Sentiment analysis
df['unique_reviews'] = df['unique_reviews'].astype(str)
df['sentiment'] = df['unique_reviews'].apply(lambda x: TextBlob(x).sentiment.polarity)

# 📊 Aggregate by Product
grouped = df.groupby('Product Name').agg({'unique_reviews': ' '.join, 'sentiment': 'mean'}).reset_index()
grouped['has_negative_sentiment'] = grouped['sentiment'] < 0

# 🔎 TF-IDF + Cosine Similarity
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(grouped['unique_reviews'])

# 🧠 Recommendation function
def recommend_reviews(selected_product):
    if not selected_product or selected_product not in grouped['Product Name'].values:
        return pd.DataFrame({'Message': ['❌ Product not found or not selected.']})

    idx = grouped[grouped['Product Name'] == selected_product].index[0]
    cosine_sim = cosine_similarity(tfidf_matrix[idx:idx+1], tfidf_matrix).flatten()
    sim_scores = sorted(enumerate(cosine_sim), key=lambda x: x[1], reverse=True)[1:6]
    indices = [i[0] for i in sim_scores]

    recommendations = grouped.iloc[indices][['Product Name', 'sentiment', 'has_negative_sentiment']].copy()
    recommendations['sentiment'] = recommendations['sentiment'].round(2)

    # 💾 Save to MySQL
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS reviewbasedrecomm;")
    cursor.execute("""
        CREATE TABLE reviewbasedrecomm (
            id INT AUTO_INCREMENT PRIMARY KEY,
            input_product VARCHAR(255),
            recommended_product VARCHAR(255),
            sentiment_score FLOAT,
            is_negative BOOLEAN
        );
    """)
    for _, row in recommendations.iterrows():
        cursor.execute("""
            INSERT INTO reviewbasedrecomm (input_product, recommended_product, sentiment_score, is_negative)
            VALUES (%s, %s, %s, %s);
        """, (selected_product, row['Product Name'], row['sentiment'], row['has_negative_sentiment']))
    conn.commit()

    return recommendations

# 🎛️ Panel UI
product_dropdown = pn.widgets.Select(name='Select Product', options=[''] + sorted(grouped['Product Name'].unique().tolist()))
output = pn.pane.DataFrame(height=300, sizing_mode='stretch_width')

def update_output(event):
    selected_product = product_dropdown.value
    recommendations = recommend_reviews(selected_product)
    output.object = recommendations

product_dropdown.param.watch(update_output, 'value')

# 🧱 Layout
pn.Column(
    "## 💬 Review-Based Product Recommender ",
    product_dropdown,
    pn.pane.Markdown("### 🧠 Recommended Products Based on Sentiment Similarity"),
    output
).servable()


C:\Users\DELL\AppData\Local\Temp\ipykernel_7232\482695013.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Column
    [0] Markdown(str)
    [1] Select(options=['', 'Algae Paper Notebook...])
    [2] Markdown(str)
    [3] DataFrame(None, height=300, sizing_mode='stretch_width')

## saving the models

In [27]:
import pickle
import os

# Create a directory to store the models if it doesn't exist
os.makedirs("../models", exist_ok=True)

# ---- From Code 1: Content-Based Recommender ----
tfidf_vectorizer_content = tfidf
tfidf_matrix_content = tfidf_matrix

with open("../models/content_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer_content, f)

with open("../models/content_tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix_content, f)

# ---- From Code 3: Hybrid Recommender ----
tfidf_vectorizer_hybrid = tfidf
tfidf_matrix_hybrid = tfidf_matrix

with open("../models/hybrid_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer_hybrid, f)

with open("../models/hybrid_tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix_hybrid, f)

# ---- From Code 5: Review-Based Recommender ----
tfidf_vectorizer_review = tfidf
tfidf_matrix_review = tfidf_matrix
grouped_review_df = grouped

with open("../models/review_tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer_review, f)

with open("../models/review_tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix_review, f)

with open("../models/review_grouped_data.pkl", "wb") as f:
    pickle.dump(grouped_review_df, f)
